# Cruzamento — Candidatos + Resultados
Fase: obter o `Status` real por (candidato, curso), a unidade correta de análise.

**Decisão de modelação confirmada:** a admissão é decidida por curso, não por candidato de forma
genérica — o mesmo candidato pode ser admitido numa opção e não admitido noutra. Por isso o
ficheiro de candidatos (formato largo: `UEM_Cod_Opc1`/`UEM_Cod_Opc2` em colunas) precisa de ser
transformado para formato longo (uma linha por candidato+curso) antes do cruzamento.

**`Obs = 'A'` significa Ausente** — candidato não compareceu ao exame. Isto é tratado como
categoria própria, nunca como "reprovado", para não confundir "não sabia a matéria" com
"não apareceu".

In [14]:
import pandas as pd

pd.set_option('display.max_columns', None)

df_candidatos = pd.read_parquet('../data/processed/candidatos_uem.parquet')
df_resultados = pd.read_excel('../data/raw/UEMResultadosFinal-APURAMENTO2026.xlsx') 

print("Candidatos:", df_candidatos.shape)
print("Resultados:", df_resultados.shape)
df_resultados.head()

Candidatos: (26798, 23)
Resultados: (51129, 16)


,Prov,NoCand,Nome,CursoID,Curso,Discip1,Nota1,Discip2,Nota2,Media,Discip3,Nota3,Obs,Resultados,Sexo,data_Nasc
0,Gaza,10019,SONIA SUARES,10100,Admin. Pública - Diurno - UEM,História-I,0.00,Português-I,0.0,0.00,NaN,NaN,A,Não admitido,F,2011-12-11
1,Gaza,10019,SONIA SUARES,10128,Arqueologia e Gest. Patr. Cultural - Diurno - UEM,Geografia-I,0.00,História-II,0.0,0.00,NaN,NaN,A,Não admitido,F,2011-12-11
2,Cidade de Maputo,10036,ELISABETH OLGA MASSANGO,11100,Engenharia Civil - Diurno - UEM,Física-I,5.40,Matemática-I,4.3,4.85,NaN,NaN,NaN,Não admitido,F,2006-04-07
3,Cidade de Maputo,10036,ELISABETH OLGA MASSANGO,11108,Engenharia Química - Diurno - UEM,Física-I,5.40,Matemática-I,4.3,4.85,NaN,NaN,NaN,Não admitido,F,2006-04-07
4,Cidade de Maputo,10037,HAWA NATURALAMA ABOO CHIRIDA,11108,Engenharia Química - Diurno - UEM,Física-I,7.84,Matemática-I,4.7,6.27,NaN,NaN,NaN,Não admitido,F,2004-12-06


## 1. Limpeza inicial do ficheiro de resultados

In [15]:
df_resultados.info()
print()
print("Valores únicos em Resultados:", df_resultados['Resultados'].unique())
print("Valores únicos em Obs:", df_resultados['Obs'].unique())

<class 'pandas.DataFrame'>
RangeIndex: 51129 entries, 0 to 51128
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Prov        51129 non-null  str           
 1   NoCand      51129 non-null  int64         
 2   Nome        51129 non-null  str           
 3   CursoID     51129 non-null  int64         
 4   Curso       51129 non-null  str           
 5   Discip1     51129 non-null  str           
 6   Nota1       51129 non-null  float64       
 7   Discip2     51129 non-null  str           
 8   Nota2       51129 non-null  float64       
 9   Media       51129 non-null  float64       
 10  Discip3     121 non-null    str           
 11  Nota3       121 non-null    str           
 12  Obs         2459 non-null   str           
 13  Resultados  51129 non-null  str           
 14  Sexo        50966 non-null  str           
 15  data_Nasc   50966 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(3),

In [16]:
# padronizar chave para bater com o tipo do ficheiro de candidatos
df_resultados['NoCand'] = df_resultados['NoCand'].astype(str).str.strip()
df_resultados['CursoID'] = df_resultados['CursoID'].astype(int)

# padronizar texto
df_resultados['Resultados'] = df_resultados['Resultados'].str.strip().str.title()
df_resultados['Obs'] = df_resultados['Obs'].fillna('').str.strip()

# criar coluna de status final considerando ausência
df_resultados['status_final'] = df_resultados.apply(
    lambda row: 'Ausente' if row['Obs'] == 'A' else row['Resultados'],
    axis=1
)
df_resultados['status_final'].value_counts()

status_final
Não Admitido    44263
Admitido         4510
Ausente          2356
Name: count, dtype: int64

## 2. Transformar candidatos de formato largo para longo
Cada candidato tem até 2 opções de curso (`UEM_Cod_Opc1`, `UEM_Cod_Opc2`). Para cruzar com os
resultados (uma linha por candidato+curso), precisamos de uma linha por opção.

In [17]:
df_candidatos['candidato_codigo'] = df_candidatos['candidato_codigo'].astype(str).str.strip()

opc1 = df_candidatos.copy()
opc1['curso_codigo'] = opc1['UEM_Cod_Opc1']
opc1['curso_nome'] = opc1['UEM_Opc1']
opc1['prioridade'] = 1

opc2 = df_candidatos[df_candidatos['UEM_Cod_Opc2'] != 0].copy()
opc2['curso_codigo'] = opc2['UEM_Cod_Opc2']
opc2['curso_nome'] = opc2['UEM_Opc2']
opc2['prioridade'] = 2

colunas_base = [c for c in df_candidatos.columns if c not in
                ['UEM_Cod_Opc1', 'UEM_Opc1', 'UEM_Cod_Opc2', 'UEM_Opc2']]

df_candidatos_longo = pd.concat([
    opc1[colunas_base + ['curso_codigo', 'curso_nome', 'prioridade']],
    opc2[colunas_base + ['curso_codigo', 'curso_nome', 'prioridade']]
], ignore_index=True)

print("Candidatos formato longo:", df_candidatos_longo.shape)
df_candidatos_longo.head()

Candidatos formato longo: (50857, 22)


,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,ProvRes,ProvCand,cod_preUni,EscoPU,local_exame,AnocPU,distrito_nascimento,distrito_residencia,data_Nasc,curso_codigo,curso_nome,prioridade
0,2026,10019,SUARES,SONIA,BILHETE DE IDENTIDADE,1233444,Feminino,Solteiro(A),Andorra,Estrangeiro,Cabo Delgado,Cidade de Maputo,6.0,Colégio Delta,Chibuto,2012,Estrangeiro,Pemba Cidade,2011-12-11,10100,Administração Pública - Diurno - Uem,1
1,2026,10039,MUARAPAZ,CLÁUDIO ARMANDO,BILHETE DE IDENTIDADE,030106033801F,Masculino,Solteiro(A),Mocambique,Nampula,Cidade de Maputo,Nampula,833.0,Escola Secundária De Teacane,Cidade De Maputo,2022,Cidade De Nampula,Kamaxaquene,2004-05-07,10300,Economia - Diurno - Uem,1
2,2026,10041,MBALANGO,YUNI ADELAIDE ANÍBAL,BILHETE DE IDENTIDADE,110107852412P,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,Província de Maputo,Cidade de Maputo,3.0,Instituto Comercial De Maputo,Cidade De Maputo,2025,Kampfumo,Matola Cidade,2007-10-13,10300,Economia - Diurno - Uem,1
3,2026,10058,SARMENTO,NENGRA AURO,BILHETE DE IDENTIDADE,100107127647P,Feminino,Solteiro(A),Mocambique,Província De Maputo,Província de Maputo,Província de Maputo,140.0,Escola Secundária De Machava Sede,Província De Maputo,2024,Matola Cidade,Matola Cidade,2007-06-27,10300,Economia - Diurno - Uem,1
4,2026,10062,SEBASTIÃO,LOIDA JOSEFA,BILHETE DE IDENTIDADE,110207815487J,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,Cidade de Maputo,Cidade de Maputo,2.0,Escola Secundária Josina Machel,Cidade De Maputo,2023,Kanlhamankulu,Kanlhamankulu,2004-09-17,10208,Desenvolvimento E Educação De Infância - Diurn...,1


## 3. Cruzamento pela chave composta (candidato + curso)

In [18]:
df_final = df_candidatos_longo.merge(
    df_resultados[['NoCand', 'CursoID', 'Nota1', 'Nota2', 'Media', 'Nota3', 'status_final']],
    left_on=['candidato_codigo', 'curso_codigo'],
    right_on=['NoCand', 'CursoID'],
    how='left'
)

print("Resultado do merge:", df_final.shape)
df_final[['candidato_codigo', 'curso_codigo', 'prioridade', 'status_final']].head(10)

Resultado do merge: (50857, 29)


,candidato_codigo,curso_codigo,prioridade,status_final
0,10019,10100,1,Ausente
1,10039,10300,1,Não Admitido
2,10041,10300,1,Admitido
3,10058,10300,1,Não Admitido
4,10062,10208,1,Não Admitido
5,10068,10302,1,Admitido
6,10079,10304,1,Não Admitido
7,10065,11100,1,Ausente
8,10066,10600,1,Não Admitido
9,10096,10106,1,Admitido


## 4. Verificar qualidade do cruzamento
Antes de seguir, precisamos de saber: quantas linhas de candidatos ficaram sem resultado
correspondente? Isto revela problemas de chave (formato diferente, candidato sem resultado
registado ainda, etc.) — não avançar sem entender esta percentagem.

In [19]:
sem_resultado = df_final['status_final'].isna().sum()
pct_sem_resultado = sem_resultado / len(df_final) * 100
print(f"Linhas candidato+curso sem resultado correspondente: {sem_resultado} ({pct_sem_resultado:.1f}%)")

# se a percentagem for alta, investigar exemplos concretos
if sem_resultado > 0:
    print()
    print("Exemplos sem correspondência:")
    print(df_final[df_final['status_final'].isna()][['candidato_codigo', 'curso_codigo', 'curso_nome']].head(10))

Linhas candidato+curso sem resultado correspondente: 47 (0.1%)

Exemplos sem correspondência:
      candidato_codigo  curso_codigo  \
1021             13602         10503   
1240             13869         11110   
3283             20381         11072   
3943             17997         10301   
4385             22937         11102   
8008             32272         10206   
10695            11984         10303   
10834            15459         11020   
12606            26532         10800   
13741            44884         10600   

                                              curso_nome  
1021      Marketing E Relações Públicas - Nocturno - Uem  
1240               Engenharia Informática - Diurno - Uem  
3283        Estatística (Mat-I E Fis-I)  - Diurno  - Uem  
3943                           Economia - Nocturno - Uem  
4385               Engenharia Electrónica - Diurno - Uem  
8008   Psicologia Escolar E De Necessidades Educativa...  
10695          Contabilidade E Finanças - Nocturno -

In [20]:
# checagem inversa: resultados que não bateram com nenhum candidato
chave_candidatos = set(zip(df_candidatos_longo['candidato_codigo'], df_candidatos_longo['curso_codigo']))
chave_resultados = set(zip(df_resultados['NoCand'], df_resultados['CursoID']))

resultados_sem_candidato = chave_resultados - chave_candidatos
print(f"Resultados sem candidato correspondente: {len(resultados_sem_candidato)}")
if resultados_sem_candidato:
    print("Exemplos:", list(resultados_sem_candidato)[:10])

Resultados sem candidato correspondente: 319
Exemplos: [('18973', 10600), ('19522', 11402), ('21222', 10506), ('11974', 10126), ('19487', 11110), ('41031', 10600), ('59266', 11116), ('64668', 11006), ('57637', 11036), ('44884', 10300)]


## 5. Distribuição final do target
Com `status_final` definido (Admitido / Não Admitido / Ausente / sem correspondência),
esta é a base real para o modelo — mas as linhas `Ausente` e sem correspondência precisam de
decisão explícita antes do treino, não devem entrar misturadas com Admitido/Não Admitido.

In [21]:
df_final['status_final'].value_counts(dropna=False)

status_final
Não Admitido    43984
Admitido         4491
Ausente          2335
NaN                47
Name: count, dtype: int64

## 6. Preparar o dataset de treino (separando ausentes)
Regra: o modelo de previsão de admissão só deve aprender com quem realmente fez a prova.
Ausentes e sem correspondência ficam guardados à parte — úteis para métricas de dashboard
(taxa de absentismo), não para o treino do classificador.

In [22]:
df_treino_ml = df_final[df_final['status_final'].isin(['Admitido', 'Não Admitido'])].copy()
df_ausentes = df_final[df_final['status_final'] == 'Ausente'].copy()
df_sem_correspondencia = df_final[df_final['status_final'].isna()].copy()

print("Base para treino do modelo:", df_treino_ml.shape)
print("Ausentes (guardados à parte):", df_ausentes.shape)
print("Sem correspondência (investigar antes de descartar):", df_sem_correspondencia.shape)

Base para treino do modelo: (48475, 29)
Ausentes (guardados à parte): (2335, 29)
Sem correspondência (investigar antes de descartar): (47, 29)


## 7. Guardar os resultados do cruzamento

In [23]:
df_final.to_parquet('candidatos_resultados_completo.parquet', index=False)
df_treino_ml.to_parquet('base_treino_ml.parquet', index=False)
df_ausentes.to_parquet('candidatos_ausentes.parquet', index=False)

print("Ficheiros guardados.")

Ficheiros guardados.


## 8. Taxa de admissão — primeira métrica real de negócio
Agora sim, com o target real, dá para responder à pergunta central do produto.

In [24]:
taxa_admissao_geral = (df_treino_ml['status_final'] == 'Admitido').mean() * 100
print(f"Taxa de admissão geral (excluindo ausentes): {taxa_admissao_geral:.1f}%")

print()
print("Taxa de admissão por curso (top 15 mais procurados):")
taxa_por_curso = df_treino_ml.groupby('curso_nome')['status_final'].apply(
    lambda x: (x == 'Admitido').mean() * 100
).round(1).sort_values(ascending=False)
print(taxa_por_curso.head(15))

Taxa de admissão geral (excluindo ausentes): 9.3%

Taxa de admissão por curso (top 15 mais procurados):
curso_nome
Ciências De Infor. Geog (Mat-I E Fis-I) - Nocturno - Uem       93.3
Geologia Marinha (Quelimane) - Diurno - Uem                    87.5
Geologia E Pesquisa Mineral - Nocturno - Uem                   87.5
Ensino De Filosofia (Por-I E His-I) - Nocturno - Uem           87.5
Arquivística (Mat-Ii E Por-Ii) - Nocturno - Uem                87.5
Engenharia Do Ambiente - Nocturno - Uem                        87.0
Produção Animal (Vilankulo) - Diurno - Uem                     86.4
Ensino De Línguas Bantu - Diurno - Uem                         85.7
Arquivística (Por-I E His-I) - Nocturno - Uem                  85.3
Química Marinha (Quelimane) - Diurno - Uem                     82.6
Hidrogeologia E Recursos Hídricos - Nocturno - Uem             82.4
Ensino De Filosofia (Fil E Por-Ii) - Nocturno - Uem            82.1
Ciências De Infor. Geog (Mat-Ii E Por-Ii)  - Nocturno - Uem    82.1
L